In [ ]:

from pyspark.sql import functions as F

# Read the existing Silver table
silver_df = spark.read.table("silver_online_retail")

# Ensure the date column is available as a date
silver_df = silver_df.withColumn(
    "invoice_date",
    F.to_timestamp("invoice_date")
)

# Create a month column for monthly analytics
silver_df = silver_df.withColumn(
    "sales_month",
    F.date_format("invoice_date", "yyyy-MM")
)

# 1. Monthly sales analytics
gold_monthly = (
    silver_df
    .groupBy("sales_month")
    .agg(
        F.count("*").alias("transaction_rows"),
        F.countDistinct("invoice").alias("distinct_invoices"),
        F.sum("quantity").alias("net_quantity"),
        F.round(F.sum("sales_amount"), 2).alias("net_sales_amount"),
        F.sum(
            F.when(F.col("is_cancellation_invoice") == True, 1)
             .otherwise(0)
        ).alias("cancellation_rows"),
        F.sum(
            F.when(F.col("is_negative_quantity") == True, 1)
             .otherwise(0)
        ).alias("negative_quantity_rows"),
        F.countDistinct("customer_id").alias("identified_customers")
    )
)

# 2. Country sales analytics
gold_country = (
    silver_df
    .groupBy("country")
    .agg(
        F.count("*").alias("transaction_rows"),
        F.countDistinct("invoice").alias("distinct_invoices"),
        F.sum("quantity").alias("net_quantity"),
        F.round(F.sum("sales_amount"), 2).alias("net_sales_amount"),
        F.countDistinct("customer_id").alias("identified_customers")
    )
)

# 3. Product sales analytics
gold_product = (
    silver_df
    .groupBy("stock_code", "description")
    .agg(
        F.count("*").alias("transaction_rows"),
        F.sum("quantity").alias("net_quantity"),
        F.round(F.sum("sales_amount"), 2).alias("net_sales_amount"),
        F.countDistinct("invoice").alias("distinct_invoices")
    )
)


gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_retail_monthly_sales")

gold_country.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_retail_country_sales")

gold_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_retail_product_sales")

print("Gold tables created successfully.")


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:

tables = [
    "gold_retail_monthly_sales",
    "gold_retail_country_sales",
    "gold_retail_product_sales"
]

for table_name in tables:
    df = spark.read.table(table_name)
    print(f"{table_name}: {df.count():,} rows")
    display(df.limit(5))


StatementMeta(, , -1, Cancelled, , Cancelled, True)